# Construyendo el Almacén de Datos de la Liga

**El Problema:**
Ustedes son los Arquitectos de Datos de una agencia internacional de *scouting*. Cada fin de semana, el sistema transaccional (OLTP) de los estadios genera un archivo plano masivo (`players_data-2025_2026.csv`) con todos los eventos registrados. 

Si el director de la agencia intenta calcular métricas históricas conectándose directamente a este archivo crudo, el sistema colapsa por redundancia y falta de indexación analítica.

**Su Misión:**
Construir una tubería **ETL (Extract, Transform, Load)** que tome este archivo transaccional y lo convierta en un **Esquema de Estrella (OLAP)** en memoria. Al final, deberán ejecutar un cubo analítico (*Roll-up*) para responder una pregunta de negocio estratégica.

In [ ]:
import pandas as pd
import numpy as np
import os

print("--- FASE 1: EXTRACT (Extracción Transaccional) ---")
# Simulamos el robot nocturno que extrae los datos de producción.
ruta_archivo = 'players_data-2025_2026.csv'

df_raw = pd.read_csv(ruta_archivo)

print("Datos crudos extraídos del sistema OLTP:")
display(df_raw.head(3))

--- FASE 1: EXTRACT (Extracción Transaccional) ---
Datos crudos extraídos del sistema OLTP:


,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,CrdY_stats_misc,CrdR_stats_misc,2CrdY,Fls,Fld,Off,Crs,Int,TklW,OG
0,1,Brenden Aaronson,us USA,"MF,FW",Leeds United,eng Premier League,24.0,2000.0,37,30,...,3,0,0,20,51,5,37,17,27,0
1,2,Jerome Abbey,eng ENG,MF,Wolves,eng Premier League,15.0,2009.0,1,0,...,0,0,0,0,0,1,0,1,0,0
2,3,Zach Abbott,eng ENG,DF,Nottingham Forest,eng Premier League,19.0,2006.0,3,2,...,0,0,0,3,1,0,3,2,4,0


---
### FASE 2: TRANSFORM (Modelado del Esquema de Estrella)

El archivo plano tiene demasiado texto mezclado con métricas. Vamos a separarlo arquitectónicamente. Primero, crearemos nuestras **Tablas de Dimensión** (El Contexto).

**Reto 1:** Extraigan los datos únicos para crear una dimensión de Equipos y una dimensión de Jugadores. Generen IDs numéricos artificiales para optimizar la base analítica.

In [ ]:
print("--- FASE 2A: TRANSFORM (Tablas de Dimensión) ---")

dim_equipo = df_raw[['Squad', 'Comp']].drop_duplicates().reset_index(drop=True)
dim_equipo['id_equipo'] = dim_equipo.index + 1
dim_equipo = dim_equipo[['id_equipo', 'Squad', 'Comp']]

dim_jugador = df_raw[['Player', 'Pos', 'Age']].drop_duplicates().reset_index(drop=True)
dim_jugador['id_jugador'] = dim_jugador.index + 1
dim_jugador = dim_jugador[['id_jugador', 'Player', 'Pos', 'Age']]

print("Dimensión Equipo (Muestra):")
display(dim_equipo.head(3))
print("\nDimensión Jugador (Muestra):")
display(dim_jugador.head(3))

--- FASE 2A: TRANSFORM (Tablas de Dimensión) ---
Dimensión Equipo (Muestra):


,id_equipo,Squad,Comp
0,1,Leeds United,eng Premier League
1,2,Wolves,eng Premier League
2,3,Nottingham Forest,eng Premier League



Dimensión Jugador (Muestra):


,id_jugador,Player,Pos,Age
0,1,Brenden Aaronson,"MF,FW",24.0
1,2,Jerome Abbey,MF,15.0
2,3,Zach Abbott,DF,19.0


Ahora construiremos el corazón del Data Warehouse: **La Tabla de Hechos (`Fact_Rendimiento`)**.
Recuerden la regla de oro: La tabla de hechos NO debe contener texto. Solo debe tener los IDs de las dimensiones que acabamos de crear y las métricas numéricas puras (goles, asistencias, minutos).

In [ ]:
print("--- FASE 2B: TRANSFORM (Tabla de Hechos) ---")

df_temp = df_raw.merge(dim_equipo, on=['Squad', 'Comp'], how='left')
df_temp = df_temp.merge(dim_jugador, on=['Player', 'Pos', 'Age'], how='left')

fact_rendimiento = df_temp[['id_jugador', 'id_equipo', 'Gls', 'Ast', 'Min']].copy()

print("Tabla de Hechos (Métrica Pura):")
display(fact_rendimiento.head(3))

--- FASE 2B: TRANSFORM (Tabla de Hechos) ---
Tabla de Hechos (Métrica Pura):


,id_jugador,id_equipo,Gls,Ast,Min
0,1,1,4,5,2449
1,2,2,0,0,17
2,3,3,0,0,164


---
### FASE 3: LOAD Y CONSULTA OLAP (Roll-up)

Nuestra arquitectura está lista. Hemos separado el almacenamiento para que sea eficiente. 

De pronto, el Director Técnico hace una pregunta que requiere un análisis multidimensional masivo (*Roll-up*):
*"¿Cuál es la Liga de fútbol que tiene la mayor cantidad de Goles Totales acumulados, y cuántos minutos en promedio juegan los jugadores de esa liga?"*

**Reto Final:** 
Para responder, no pueden usar el archivo `df_raw`. Deben simular el comportamiento de un motor OLAP: unan la Tabla de Hechos con la Dimensión Equipos y ejecuten una agrupación.

In [ ]:
print("--- FASE 3: CUBO OLAP (Roll-up Analítico) ---")

cubo_olap = fact_rendimiento.merge(dim_equipo, on='id_equipo', how='inner')

analisis_ligas = cubo_olap.groupby('Comp').agg(
    goles_totales=('Gls', 'sum'),
    minutos_promedio=('Min', 'mean')
).sort_values(by='goles_totales', ascending=False).reset_index()

print("Respuesta Analítica para el Director Técnico:")
display(analisis_ligas)

liga_lider = analisis_ligas.iloc[0]
print("\nRespuesta para el Director Técnico:")
print(f"- Liga con mayor cantidad de Goles Totales acumulados: {liga_lider['Comp']} ({int(liga_lider['goles_totales'])} goles).")
print(f"- Minutos promedio jugados por los futbolistas en esa liga: {liga_lider['minutos_promedio']:.2f} minutos.")

--- FASE 3: CUBO OLAP (Roll-up Analítico) ---
Respuesta Analítica para el Director Técnico:


,Comp,goles_totales,minutos_promedio
0,eng Premier League,1005,1362.834846
1,es La Liga,1001,1227.878887
2,de Bundesliga,970,1192.455621
3,it Serie A,900,1232.576355
4,fr Ligue 1,838,1075.185383



Respuesta para el Director Técnico:
- Liga con mayor cantidad de Goles Totales acumulados: eng Premier League (1005 goles).
- Minutos promedio jugados por los futbolistas en esa liga: 1362.83 minutos.


---
### 🧠 Defensa del Diseño Arquitectónico (Responde aquí)

Como ingenieros de datos, acaban de transformar un archivo plano en un modelo de estrella y luego volvieron a unir las tablas para hacer la consulta OLAP. 

1. Si al final tuvimos que hacer un `merge()` (JOIN) para responder la pregunta del Director, ¿por qué nos tomamos la molestia de separar los datos en Hechos y Dimensiones en lugar de dejar el `df_raw` gigante en el servidor analítico desde el principio? 
2. ¿Qué ventaja física (de disco o memoria) tiene aislar las columnas de texto (`player_name`, `team_name`) en dimensiones separadas?

> **Tu Respuesta:**


### Respuesta 1

Nos tomamos la molestia de separar los datos porque el archivo df_raw contiene 102 columnas mezcladas, y dejarlo completo en el servidor analítico obligaría a escanear y cargar en memoria un volumen masivo de datos innecesarios en cada consulta; en cambio, con el esquema en estrella aplicamos proyección selectiva uniendo únicamente la dimensión requerida (dim_equipo), resolvemos el merge() mediante identificadores enteros que la CPU procesa órdenes de magnitud más rápido que comparar cadenas de texto repetidas, habilitamos la flexibilidad multidimensional del motor OLAP (Roll-up y Drill-down) y garantizamos la mantenibilidad del sistema al poder actualizar información de equipos o jugadores en una sola fila sin alterar millones de registros transaccionales históricos.

### Respuesta 2

Aislar las columnas de texto en dimensiones independientes elimina la redundancia física al almacenar las cadenas de caracteres una sola vez, lo que permite que la tabla de hechos contenga exclusivamente valores numéricos homogéneos de longitud fija (4 u 8 bytes) y reduzca la huella en memoria y disco entre un 70% y 80%; además, al estructurar datos numéricos contiguos se maximiza la localidad espacial en la memoria caché del procesador (L1/L2/L3), evitando los continuos fallos de lectura (cache misses) ocasionados por desreferenciar punteros a cadenas de texto dispersas en RAM, lo que habilita a la CPU para procesar métricas en paralelo mediante instrucciones vectoriales (SIMD) y permite a los motores analíticos aplicar algoritmos de compresión columnar de altísima eficiencia.